# Free video generation on Colab

Runs open-source AI video models on Google's **free** GPU. No debit/credit card — just a Google account.

**Honest limits (read once):**
- Clips are **short** (a few seconds) and come out at ~**480–720p**; we upscale to **1080p** as a second step.
- About **1–5 minutes per clip** once the model is loaded.
- Quality is **decent, not the paid tools** (Runway/Kling). It's real generation, just the free tier.
- Free Colab disconnects after a few hours and has a rough weekly limit. Re-open and re-run when you want more.

**There is no single 'watch my video and remake it' button** anywhere, free or paid, that runs on a free GPU.
What this does instead: generate a fresh clip **from a text prompt**, or **animate a still** — including a frame
pulled straight out of your own video. That's the realistic version of 'same idea, same motion, from scratch.'


## How to run
1. **Runtime → Change runtime type → T4 GPU → Save.** (Free. Without this, nothing below works.)
2. Run each cell top to bottom with **Shift+Enter**.
3. The first model cell downloads a few GB — slow *once*. Mount Drive (next cell) to cache it so future runs are fast.
4. Pick the path you want: **Text → video**, or **Image → video** (optionally from your own video's frame).


In [ ]:
#@title 1. Check the GPU  { display-mode: "form" }
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'no nvidia-smi')
assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then re-run.')
print('CUDA OK:', torch.cuda.get_device_name(0))


In [ ]:
#@title 2. Install the libraries  { display-mode: "form" }
# Quiet install. Takes ~1-2 min the first time.
!pip -q install -U diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece
print('installed')


In [ ]:
#@title 3. (Recommended) Mount Drive so models cache + videos save there  { display-mode: "form" }
# Run this BEFORE the model cells. It parks the multi-GB model download in your
# Drive, so the next session doesn't re-download it, and saves outputs to Drive.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/colab_video/hf_cache'
    OUT = '/content/drive/MyDrive/colab_video/out'
except Exception as e:
    print('Drive not mounted (%s) - using local /content, nothing persists.' % e)
    os.environ['HF_HOME'] = '/content/hf_cache'
    OUT = '/content/out'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('model cache ->', os.environ['HF_HOME'])
print('outputs   ->', OUT)


## Path A — Text → video (CogVideoX-2B)
Type what you want. Good for a brand-new clip from an idea. ~6s, 720×480, fits the free T4.


In [ ]:
#@title A. Generate from a prompt  { display-mode: "form" }
prompt = "A red fox trotting through a snowy pine forest at dawn, soft light, cinematic, smooth motion" #@param {type:"string"}
num_frames = 49 #@param {type:"integer"}
steps = 50 #@param {type:"integer"}
fps = 8 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}

import os, torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

pipe = CogVideoXPipeline.from_pretrained('THUDM/CogVideoX-2b', torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()      # streams the model through the GPU: fits 16GB
pipe.vae.enable_tiling()             # lets the decoder fit too

video = pipe(prompt=prompt, num_frames=num_frames, num_inference_steps=steps,
             guidance_scale=6.0,
             generator=torch.Generator(device='cpu').manual_seed(seed)).frames[0]
dst = os.path.join(OUT, 'text2video.mp4')
export_to_video(video, dst, fps=fps)
print('saved', dst)
from IPython.display import Video; Video(dst, embed=True, width=480)


## Path B — Image → video (Stable Video Diffusion)
Give it a still and it adds motion. This is the closest free thing to 'take my video's look and move it.'

**B1** (optional): pull a frame out of *your own* video to use as the still.  
**B2**: animate a still (the frame from B1, or any image you upload).


In [ ]:
#@title B1. (Optional) Pull a frame from your video  { display-mode: "form" }
which_frame = "first" #@param ["first", "middle"]
from google.colab import files
up = files.upload()                       # choose your source video
vid = list(up.keys())[0]
import subprocess
if which_frame == 'middle':
    dur = float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration',
        '-of','default=nk=1:nw=1', vid], capture_output=True, text=True).stdout.strip() or 0)
    sel = '-ss ' + str(dur/2)
else:
    sel = ''
!ffmpeg -y {sel} -i "$vid" -vframes 1 /content/source_frame.png -loglevel error
print('frame -> /content/source_frame.png  (use it as the image in B2)')
from IPython.display import Image as _Img; _Img('/content/source_frame.png', width=360)


In [ ]:
#@title B2. Animate a still into a clip  { display-mode: "form" }
use_frame_from_B1 = True #@param {type:"boolean"}
num_frames = 25 #@param {type:"integer"}
fps = 7 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}

import os, torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
from PIL import Image

if use_frame_from_B1 and os.path.exists('/content/source_frame.png'):
    img_path = '/content/source_frame.png'
else:
    from google.colab import files
    img_path = list(files.upload().keys())[0]
image = Image.open(img_path).convert('RGB').resize((1024, 576))

pipe = StableVideoDiffusionPipeline.from_pretrained(
    'stabilityai/stable-video-diffusion-img2vid-xt',
    torch_dtype=torch.float16, variant='fp16')
pipe.enable_model_cpu_offload()

frames = pipe(image, decode_chunk_size=2, num_frames=num_frames,
              generator=torch.manual_seed(seed)).frames[0]
dst = os.path.join(OUT, 'image2video.mp4')
export_to_video(frames, dst, fps=fps)
print('saved', dst)
from IPython.display import Video; Video(dst, embed=True, width=480)


## Finish — upscale to 1080p and (optionally) smooth it
Point `src_name` at whichever clip you made (`text2video.mp4` or `image2video.mp4`).


In [ ]:
#@title Upscale to 1080p  { display-mode: "form" }
src_name = "text2video.mp4" #@param ["text2video.mp4", "image2video.mp4"]
import os
src = os.path.join(OUT, src_name)
dst = os.path.join(OUT, '1080p_' + src_name)
!ffmpeg -y -i "$src" -vf "scale=-2:1080:flags=lanczos" -c:v libx264 -crf 18 -pix_fmt yuv420p "$dst" -loglevel error
print('saved', dst)
from IPython.display import Video; Video(dst, embed=True, width=480)


In [ ]:
#@title (Optional) Smooth motion to 30fps  { display-mode: "form" }
# Interpolates in-between frames so the short clip looks less choppy.
src_name = "1080p_text2video.mp4" #@param {type:"string"}
import os
src = os.path.join(OUT, src_name)
dst = os.path.join(OUT, 'smooth_' + src_name)
!ffmpeg -y -i "$src" -vf "minterpolate=fps=30:mi_mode=mci:mc_mode=aobmc:vsbmc=1" -c:v libx264 -crf 18 -pix_fmt yuv420p "$dst" -loglevel error
print('saved', dst)
from IPython.display import Video; Video(dst, embed=True, width=480)


In [ ]:
#@title Download a result to your computer  { display-mode: "form" }
file_name = "smooth_1080p_text2video.mp4" #@param {type:"string"}
import os
from google.colab import files
files.download(os.path.join(OUT, file_name))


## If something goes wrong
- **`CUDA out of memory`** — lower `num_frames` (e.g. 25 for text, 14 for image) and re-run. The model cell can be
  re-run after an OOM without restarting.
- **Model download is huge / slow** — that's the one-time cost; mount Drive (cell 3) so it's cached next time.
- **`variant='fp16'` error on SVD** — delete `, variant='fp16'` from the `from_pretrained(...)` line and re-run.
- **Disconnected / 'GPU limit reached'** — free-tier limit. Wait a while or come back tomorrow; mounted Drive keeps your cache and outputs.
- Models used: `THUDM/CogVideoX-2b` (text→video) and `stabilityai/stable-video-diffusion-img2vid-xt` (image→video).
